### Retrieve an ET time series using the OpenET API

This notebook guides you through the process of accessing and visualizing a time series of OpenET  data via from a point.


#### Get Set Up ###
* Import necessary Python libraries
* Load your API key.



In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import getpass

* Find your API Key from the [OpenET account](https://auth.etdata.org/).
* Copy your API Key to paste below.

In [ ]:
YOUR_API_KEY = input("Paste your API key here: ")
# set your API key before making the request
header = {"Authorization": YOUR_API_KEY}

#### Construct argument to pass to the API.

* To get data from the OpenET API, certain parameters are required including the **date_range**, **interval**, **geometry**, **model**, **variable**, **units** and **reference_et**.

* For more information on appropriate parameters to pass to the API end point: https://openet.gitbook.io/docs/reference/api-reference#accepted-parameters

In [ ]:
# arguments to pass to API endpoint
args = {
    "date_range": [
        "2016-01-01",        # start date
        "2020-12-31"         # end date
    ],
    "interval": "daily",     # frequency of data: daily or monthly
    "geometry": [
      -121.6467, 38.1159     # coordinates in decimal degrees
    ],
    "model": "Ensemble",     # Model options: Ensemble, geeSEBAL, SSEBop, SIMS, DisALEXI, PTJPL, or eeMETRIC.
    "overpass":False,        # Optional parameter for daily timesteps return only satellite overpass dates
    "variable": "ET",        # Variable options: ET, EToF, ETo, PR, NDVI
    "reference_et": "CIMIS", # ETo options: CIMiS, gridMET
    "units": "mm",           # Unit options: mm, in
    "file_format": "JSON",    # JSON, CSV
    "version": 2.1
}

### Send Request to API
* Pass API key and the arguments to the `raster/timeseries/point` API endpoint.
* Retrieve data to variable *output*

In [ ]:
# query the api
resp = requests.post(
    headers=header,
    json=args,
    url="https://california.openet-api.org/raster/timeseries/point"
)
# store the result
if resp.ok:
    output = resp.json()
else:
    print(resp.status_code)
    print(resp.text)

### Save data as a csv file and format data for analysis

* Format the response as a dataframe with a time index
* Print first 5 rows to inspect data
* Save file to content/ folder

In [ ]:
# covert to a dataframe
df = pd.DataFrame(output)
df["time"] = pd.to_datetime(df["time"])
df.set_index("time", inplace=True)

# save dataframe to a csv file
df.to_csv('openet.csv')

### Plot the ET data
* An example of how to plot the time series usint matplotlib

In [ ]:
plt.style.use("ggplot")
plt.figure(figsize=(6, 3))
df["et"].plot(c='#008fd5',style='o',ms=3, label="OpenET Ensemble")
plt.xlabel(" ")
plt.ylabel("ET (mm)")
plt.legend(frameon=False)
plt.show()
plt.tight_layout()

### Bonus Section: Request reference ET data (ETo) and compare with ETa

This section builds from the query structure above:

* Change the `variable` from `ET` to `ETo`

#### Retrieve the ETo data from the API

In [ ]:

# arguments to pass to API endpoint
ETo_args = {
    "date_range": [
        "2016-01-01",        
        "2020-12-31"         
    ],
    "interval": "daily",     
    "geometry": [
      -121.6467, 38.1159     
    ],
    "model": "Ensemble",     
    "variable": "ETo",        # <-------- Change variable to ETo
    "reference_et": "CIMIS", # ETo options CIMiS, gridMET
    "units": "mm",           
    "file_format": "JSON"    
}
# Send Request to API
# Pass API key and the arguments to the `raster/timeseries/point` API endpoint.
# query the api
ETo_resp = requests.post(
    headers=header,
    json=ETo_args,
    url="https://california.openet-api.org/raster/timeseries/point"
)
# store the result
ETo_output = ETo_resp.json()

In [ ]:
# covert to a dataframe save to different file name
df_ETo = pd.DataFrame(ETo_output)
df_ETo["time"] = pd.to_datetime(df_ETo["time"])
df_ETo.set_index("time", inplace=True)

# save dataframe to a csv file
df_ETo.to_csv('openet_CIMIS_ETo.csv')

#### Plot the ETo data from CIMIS

In [ ]:
plt.style.use("ggplot")
plt.figure(figsize=(6, 3))
df_ETo["eto"].plot(c='forestgreen',style='o',ms=3, label="CIMIS ETo")
plt.xlabel(" ")
plt.ylabel("ET (mm)")
plt.legend(frameon=False)
plt.show()
plt.tight_layout()

#### Plot data together on the same figure

In [ ]:
plt.style.use("tableau-colorblind10")
plt.figure(figsize=(6, 3))
df_ETo["eto"].plot(c='forestgreen',style='o',ms=1, label="CIMIS ETo")
df["et"].plot(c='#008fd5',style='o',ms=1, label="OpenET Ensemble")
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2)

plt.xlabel(" ")
plt.ylabel("ET | ETo (mm)")
plt.show()
plt.tight_layout()